# 01 — Genuine Sentinel-2 Patch Extraction
Vellakkal Dumpsite methane hotspot project.

This notebook drives `src/gee_extraction.py` to pull genuine 32x32x7 Sentinel-2 patches from Earth Engine for every valid (lat, lon, date) row in the original CSV, export them to Google Drive as GeoTIFFs, and log provenance (scene ID, matched date, cloud %, export status).

**No band values are copied from the CSV** — everything is freshly queried from GEE.

In [1]:
# Mount Drive (exports land here) and clone/attach project source
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/methane-hotspot-detection'
import os
os.makedirs(f'{PROJECT_ROOT}/data/raw', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/src', exist_ok=True)

MessageError: Error: credential propagation was unsuccessful

In [ ]:
!pip install -q earthengine-api geemap tqdm

In [ ]:
import ee
GEE_PROJECT = 'YOUR-GEE-PROJECT-ID'  # <-- set this
try:
    ee.Initialize(project=GEE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)

## Upload the original CSV
Place `original_gee_export.csv` (the 6,200-row export) into `data/raw/` before running the next cell.

In [ ]:
CSV_PATH = f'{PROJECT_ROOT}/data/raw/original_gee_export.csv'
import pandas as pd
df = pd.read_csv(CSV_PATH)
print(df.shape)
df.head()

## Sanity-check test run
Run on a small subset first (`--limit 20`) to confirm scenes are found and exports submit cleanly before committing to the full 6,200-row run.

In [ ]:
import sys
sys.path.append(f'{PROJECT_ROOT}/src')

!python {PROJECT_ROOT}/src/gee_extraction.py \
    --csv {CSV_PATH} \
    --drive-folder methane_patches_test \
    --patch-size 32 \
    --gee-project {GEE_PROJECT} \
    --out-log {PROJECT_ROOT}/data/raw/extraction_log_test.csv \
    --limit 20

In [ ]:
test_log = pd.read_csv(f'{PROJECT_ROOT}/data/raw/extraction_log_test.csv')
test_log['status'].value_counts()

## Preview a sample patch footprint on a map (visual QA)
Confirms the 32x32 window and matched scene look correct before running the full extraction.

In [ ]:
import geemap
from gee_extraction import find_best_scene, make_patch_region, build_patch_image

row = df.iloc[0]
scene = find_best_scene(row['longitude'], row['latitude'], str(row['date'])[:10])
region = make_patch_region(row['longitude'], row['latitude'], 32)
patch_img = build_patch_image(scene)

m = geemap.Map(center=[row['latitude'], row['longitude']], zoom=16)
m.addLayer(patch_img, {'bands': ['B4','B3','B2'], 'min': 0, 'max': 3000}, 'True color')
m.addLayer(region, {}, 'Patch footprint')
m

## Full extraction run
This submits export tasks for the entire CSV. Exports run server-side on Earth Engine and are asynchronous — this cell only *submits* tasks, it does not wait for completion.

In [ ]:
!python {PROJECT_ROOT}/src/gee_extraction.py \
    --csv {CSV_PATH} \
    --drive-folder methane_patches \
    --patch-size 32 \
    --gee-project {GEE_PROJECT} \
    --out-log {PROJECT_ROOT}/data/raw/extraction_log.csv

## Poll export tasks until completion
Earth Engine exports can take from minutes to hours depending on queue load. This cell blocks and checks task status periodically, then updates the log with final COMPLETED/FAILED states.

In [ ]:
from gee_extraction import check_export_status

log_df = pd.read_csv(f'{PROJECT_ROOT}/data/raw/extraction_log.csv')
log_df = check_export_status(log_df, poll_seconds=30)
log_df.to_csv(f'{PROJECT_ROOT}/data/raw/extraction_log.csv', index=False)
log_df['status'].value_counts()

## Result summary
Report the true number of genuine patches obtained (COMPLETED exports) — this is the honest count that feeds Batch 2 (CH4 labeling) and beyond. No padding or duplication happens here or later.

In [ ]:
n_completed = (log_df['status'] == 'COMPLETED').sum()
print(f'Genuine patches successfully exported: {n_completed}')